# Fase 1b: Ingesta Vectorial y Embeddings (Entorno GPU)

## 1. Introducción
Este notebook clona las noticias ya limpias del índice base (`noticias_tfg`) para pasarlas por un modelo de embeddings en la GPU y guardarlas en un nuevo índice (`noticias_tfg_vectores`). 

## 2. Decisiones Arquitectónicas y de Seguridad
* **Protección de Almacenamiento:** Debido a la limitación de espacio en `valencia2`, no se procesa el dataset original. Se leen los datos limpios directamente desde `valencia` a través del túnel SSH.
* **Idempotencia:** Se preserva el `_id` interno de Elasticsearch de cada noticia. Si el proceso se interrumpe y se reinicia, los vectores se sobrescriben en lugar de duplicarse.
* **Modelo Multilingüe:** Se emplea `paraphrase-multilingual-MiniLM-L12-v2` para manejar correctamente noticias que contengan anglicismos o texto en otros idiomas.

In [1]:
from elasticsearch import Elasticsearch

# Conexión a través del túnel SSH hacia valencia
ES_HOST = "http://localhost:9250"
INDEX_ORIGEN = "noticias_tfg"
INDEX_DESTINO = "noticias_tfg_vectores"

print(f"Conectando a Elasticsearch en {ES_HOST}...")
es = Elasticsearch(ES_HOST)

if es.ping():
    print(f"Conexión exitosa. Versión: {es.info()['version']['number']}")
else:
    print("Error: No se puede conectar. ¿Está abierto el túnel SSH desde valencia2?")

Conectando a Elasticsearch en http://localhost:9250...
Conexión exitosa. Versión: 8.12.0


In [7]:
import torch
from sentence_transformers import SentenceTransformer

print("Cargando modelo de Embeddings (8k tokens)...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

modelo_embeddings = SentenceTransformer('BAAI/bge-m3', device=device)
print(f"Modelo cargado en: {device.upper()}")

Cargando modelo de Embeddings (8k tokens)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2624.17it/s, Materializing param=pooler.dense.weight]                               


Modelo cargado en: CUDA


In [8]:
# Si existe lo borramos para empezar de cero limpio (opcional, pero recomendado en la primera ejecución)
if es.indices.exists(index=INDEX_DESTINO):
    es.indices.delete(index=INDEX_DESTINO)
    print(f"Índice anterior '{INDEX_DESTINO}' eliminado.")

# Mapping idéntico al original, pero con la suma del vector matemático
settings = {
    "mappings": {
        "properties": {
            "title": {"type": "text", "analyzer": "spanish"},
            "body":  {"type": "text", "analyzer": "spanish"},
            "date":  {"type": "date", "ignore_malformed": True}, 
            "url":   {"type": "keyword", "ignore_above": 256}, 
            "source":{"type": "keyword"},
            "vector_texto": {
                "type": "dense_vector",
                "dims": 1024,
                "index": True,         
                "similarity": "cosine" 
            }
        }
    }
}

es.indices.create(index=INDEX_DESTINO, body=settings)
print(f"Índice '{INDEX_DESTINO}' preparado.")

Índice 'noticias_tfg_vectores' preparado.


In [9]:
from elasticsearch import helpers
from tqdm import tqdm
import time

def generar_vectores_desde_origen():
    """
    Lee usando la API 'scan' de Elasticsearch, calcula el vector y prepara el guardado.
    """
    # 1. Obtenemos el total de documentos para la barra de progreso
    total_docs = es.count(index=INDEX_ORIGEN)['count']
    print(f"Se van a procesar {total_docs} documentos desde '{INDEX_ORIGEN}'...")
    
    # 2. helpers.scan es la forma segura de extraer datos masivos sin saturar RAM
    cursor = helpers.scan(
        es,
        index=INDEX_ORIGEN,
        query={"query": {"match_all": {}}}, # Traer todo
        _source=["title", "body", "date", "url", "source"] # Solo traemos lo que necesitamos
    )
    
    for doc in tqdm(cursor, total=total_docs, desc="Vectorizando"):
        doc_id = doc["_id"] # <-- ESTA ES LA CLAVE ANTIDUPLICADOS
        source_data = doc["_source"]
        
        titulo = source_data.get("title", "")
        cuerpo = source_data.get("body", "")
        
        # Unimos para darle máximo contexto al LLM
        texto_completo = f"{titulo}. {cuerpo}"
        
        # Generamos el vector en la GPU
        vector = modelo_embeddings.encode(texto_completo).tolist()
        
        # Preparamos la acción de escritura
        yield {
            "_op_type": "index",      # Si el ID ya existe, lo sobreescribe
            "_index": INDEX_DESTINO,
            "_id": doc_id,            # Mantenemos el ID original
            "_source": {
                "title": titulo,
                "body": cuerpo,
                "date": source_data.get("date", ""),
                "url": source_data.get("url", ""),
                "source": source_data.get("source", "Desconocido"),
                "vector_texto": vector
            }
        }

# --- EJECUCIÓN ---
print("\nIniciando Ingesta Vectorial...")
start_time = time.time()

# Chunk_size=400 envía paquetes pequeños de vectores por el túnel para no colapsar la red ni la memoria
success, errors = helpers.bulk(
    es, 
    generar_vectores_desde_origen(), 
    raise_on_error=False, 
    stats_only=False,
    chunk_size=400 
)

duration = time.time() - start_time

print(f"\nPROCESO FINALIZADO en {duration/60:.2f} minutos.")
print(f"Éxitos: {success}")
print(f"Fallos: {len(errors)}")


Iniciando Ingesta Vectorial...
Se van a procesar 490956 documentos desde 'noticias_tfg'...


Vectorizando: 100%|██████████| 490956/490956 [2:57:52<00:00, 46.00it/s]   



PROCESO FINALIZADO en 177.89 minutos.
Éxitos: 490956
Fallos: 0


In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9250")
index_name = "noticias_tfg_vectores"

print(" INICIANDO AUDITORÍA DEL ÍNDICE VECTORIAL...\n")

# 1. Comprobar el número exacto de documentos
count = es.count(index=index_name)['count']
print(f"Total de documentos en '{index_name}': {count}")
if count == 490956:
    print("MATCH PERFECTO con el índice original.")

# 2. Extraer un documento para inspeccionar sus "tripas"
res = es.search(index=index_name, query={"match_all": {}}, size=1)

if res['hits']['hits']:
    doc = res['hits']['hits'][0]['_source']
    print(f"\nTítulo de muestra: {doc.get('title', 'Sin título')}")
    print(f"Fuente: {doc.get('source', 'Desconocido')}")
    
    # 3. La prueba de fuego: ¿Está el vector y mide lo que tiene que medir?
    if 'vector_texto' in doc:
        longitud_vector = len(doc['vector_texto'])
        print(f"\nEl campo 'vector_texto' está guardado.")
        print(f"Longitud del vector: {longitud_vector} dimensiones (Debe ser 1024 para BGE-M3).")
        print(f"Primeros 5 números del embedding: {doc['vector_texto'][:5]}")
    else:
        print("\nError: El documento no tiene el campo 'vector_texto'.")
else:
    print("\nError: El índice está vacío.")

 INICIANDO AUDITORÍA DEL ÍNDICE VECTORIAL...

Total de documentos en 'noticias_tfg_vectores': 490956
MATCH PERFECTO con el índice original.

Título de muestra: El fuego interior de Franz Kafka
Fuente: El Universal

El campo 'vector_texto' está guardado.
Longitud del vector: 1024 dimensiones (Debe ser 1024 para BGE-M3).
Primeros 5 números del embedding: [-0.009408465586602688, 0.01189308613538742, 0.002267782110720873, -0.030145099386572838, -0.03247831389307976]
